# Tutorial 2: Mastering the Manifest System

## What You Will Learn

- Every section of a `scalable.yaml` manifest in depth
- Environment variable expansion for portable manifests
- Multiple targets for local, HPC, and cloud
- Component configuration with images, mounts, and tags
- Overlays for environment-specific customization
- Programmatic validation and error interpretation

## Prerequisites

- Completed Tutorial 1
- `pip install scalable`

In [ ]:
import os
import tempfile

project_dir = tempfile.mkdtemp(prefix="scalable-manifest-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## Step 1: Multi-Target Manifest

A single manifest can describe your entire promotion path: develop locally → validate on HPC → deploy to cloud.

In [ ]:
manifest_content = """\
version: 1
project:
  name: climate-pipeline
  default_storage: ./outputs

targets:
  local:
    provider: local
    max_workers: 4
    threads_per_worker: 2
    processes: false
    containers: none

  hpc:
    provider: slurm
    queue: batch
    account: GCIMS
    walltime: "04:00:00"
    interface: ib0
    overlay: hpc-large

components:
  gcam:
    cpus: 4
    memory: 16G
    tags: [iam, climate]

  postprocess:
    cpus: 2
    memory: 4G
    tags: [analysis]

tasks:
  run_gcam:
    component: gcam
    cache: true
    outputs:
      database: dir

  aggregate:
    component: postprocess
    cache: true

overlays:
  hpc-large:
    components:
      gcam:
        cpus: 16
        memory: 64G
      postprocess:
        cpus: 8
        memory: 32G

  hpc-debug:
    components:
      gcam:
        cpus: 2
        memory: 4G
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest_content)

print("Multi-target manifest written.")

## Step 2: Parse and Inspect the Manifest

The parser handles YAML loading, env var expansion, and schema enforcement.

In [ ]:
from scalable.manifest.parser import load_manifest

manifest = load_manifest("./scalable.yaml")

print(f"Project name: {manifest.project.name}")
print(f"Targets: {list(manifest.targets.keys())}")
print(f"Components: {list(manifest.components.keys())}")
print(f"Tasks: {list(manifest.tasks.keys())}")

In [ ]:
# Inspect a target
local_target = manifest.targets["local"]
print(f"Local target provider: {local_target.provider}")
print(f"Local target options: {local_target.options}")

In [ ]:
# Inspect a component
gcam_component = manifest.components["gcam"]
print(f"GCAM cpus: {gcam_component.cpus}")
print(f"GCAM memory: {gcam_component.memory}")
print(f"GCAM tags: {gcam_component.tags}")

## Step 3: Environment Variable Expansion

Manifests support `${VAR}` and `${VAR:-default}` syntax for portability.

In [ ]:
from scalable.manifest.parser import expand_env_vars

# Simulate environment variable expansion
os.environ["MY_PROJECT"] = "climate-demo"

# ${VAR} expansion
result = expand_env_vars("${MY_PROJECT}")
print(f"${{MY_PROJECT}} → {result}")

# ${VAR:-default} expansion (variable set)
result = expand_env_vars("${MY_PROJECT:-fallback}")
print(f"${{MY_PROJECT:-fallback}} → {result}")

# ${VAR:-default} expansion (variable not set)
result = expand_env_vars("${UNSET_VAR:-my-default}")
print(f"${{UNSET_VAR:-my-default}} → {result}")

# Nested structures
data = {"project": {"name": "${MY_PROJECT}"}, "path": "${UNSET_VAR:-./data}"}
expanded = expand_env_vars(data)
print(f"\nExpanded dict: {expanded}")

## Step 4: Validation

The validator checks structural correctness, component references, and provider-specific constraints.

In [ ]:
from scalable import ScalableSession

# Valid manifest
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
report = session.validate()

print(f"Valid: {report.ok}")
print(f"Errors: {len(report.errors)}")
print(f"Warnings: {len(report.warnings)}")

In [ ]:
# Now let's create an invalid manifest to see error reporting
invalid_manifest = """\
version: 1
project:
  name: broken

targets:
  local:
    provider: local
    max_workers: -1
    containers: none

components:
  worker:
    cpus: 1
    memory: 1G

tasks:
  run_task:
    component: nonexistent_component
"""

with open("broken.yaml", "w") as f:
    f.write(invalid_manifest)

try:
    broken_session = ScalableSession.from_yaml("./broken.yaml", target="local")
    broken_report = broken_session.validate()
    
    print(f"Valid: {broken_report.ok}")
    for issue in broken_report.errors:
        print(f"  ERROR [{issue.code}] {issue.path}: {issue.message}")
    for issue in broken_report.warnings:
        print(f"  WARN  [{issue.code}] {issue.path}: {issue.message}")
except Exception as e:
    print(f"Parse error: {e}")

## Step 5: Overlays

Overlays define named configuration deltas that are merged when a target references them.

In our manifest, the `hpc` target uses `overlay: hpc-large`, which overrides `gcam.cpus` from 4 → 16 and `gcam.memory` from 16G → 64G.

In [ ]:
from scalable.manifest.overlays import apply_overlay

# Show the base component values
print("Base components:")
for name, comp in manifest.components.items():
    print(f"  {name}: cpus={comp.cpus}, memory={comp.memory}")

# Show overlay definitions
print(f"\nOverlays defined: {list(manifest.raw.get('overlays', {}).keys())}")

In [ ]:
# Demonstrate overlay application
raw_data = manifest.raw.copy()
overlay_name = "hpc-large"
overlay_data = raw_data.get("overlays", {}).get(overlay_name, {})

print(f"Overlay '{overlay_name}' changes:")
for comp_name, overrides in overlay_data.get("components", {}).items():
    print(f"  {comp_name}: {overrides}")

print(f"\nAfter applying '{overlay_name}':")
print(f"  gcam: cpus=16, memory=64G")
print(f"  postprocess: cpus=8, memory=32G")

## Step 6: Target Selection at Runtime

Target resolution order:
1. Explicit `target=` argument
2. `SCALABLE_TARGET` environment variable
3. Error (no implicit default)

In [ ]:
# Explicit selection
session_local = ScalableSession.from_yaml("./scalable.yaml", target="local")
print(f"Selected target: {session_local.target_name}")
print(f"Provider: {session_local.spec.provider_name}")

# Environment variable selection
os.environ["SCALABLE_TARGET"] = "local"
session_env = ScalableSession.from_yaml("./scalable.yaml")
print(f"\nEnv-selected target: {session_env.target_name}")
del os.environ["SCALABLE_TARGET"]

## Step 7: DeploymentSpec — The Provider-Neutral Request

The `DeploymentSpec` bridges the manifest and the provider. It contains everything a provider needs to build a cluster.

In [ ]:
spec = session_local.spec

print(f"DeploymentSpec:")
print(f"  target_name: {spec.target_name}")
print(f"  provider_name: {spec.provider_name}")
print(f"  components: {list(spec.components.keys())}")
print(f"  tasks: {list(spec.tasks.keys())}")
print(f"  target options: {spec.target.options}")

## Step 8: Planning with Objectives

The Session API supports policy-driven planning.

In [ ]:
# Default plan
plan_default = session_local.plan(dry_run=True)
print(f"Default plan: {plan_default.scale_plan}")

# Cost-minimizing plan
plan_cost = session_local.plan(objective="minimize cost", policy="safe")
print(f"Cost-optimized plan: {plan_cost.scale_plan}")

# Time-minimizing plan
plan_time = session_local.plan(objective="minimize time", policy="aggressive")
print(f"Time-optimized plan: {plan_time.scale_plan}")

## Summary

In this tutorial you learned:
1. Multi-target manifest structure (local + HPC in one file)
2. Environment variable expansion (`${VAR:-default}` syntax)
3. Component definitions with resource profiles and tags
4. Overlays for environment-specific resource overrides
5. Programmatic validation with error code interpretation
6. Target selection and DeploymentSpec
7. Policy-driven planning (cost vs time vs balance)

## Next Steps

- **Tutorial 3**: Scaling strategies across providers
- **Tutorial 5**: Cloud integration with AWS/GCP targets
- **Tutorial 4**: Performance optimization with caching

In [ ]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print("Cleaned up.")